In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from tqdm import tqdm


In [2]:
import transformers
print(transformers.__file__)
print(transformers.__version__)


/home/jawwad/repos/Translation/env/lib64/python3.14/site-packages/transformers/__init__.py
5.1.0


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)


Using device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
VRAM (GB): 3.951296512


In [3]:
model_name = "google/mt5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

base_model.to(device)
base_model.eval()  # we are NOT training full mt5


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

MT5ForConditionalGeneration(
  (shared): Embedding(250112, 512)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 512)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
          

In [4]:
dataset = load_dataset("csv", data_files="newspaper_en_hi_translated.csv")
print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['english', 'hindi'],
        num_rows: 500
    })
})


In [5]:
MAX_LEN = 64

def preprocess(example):
    model_inputs = tokenizer(
        example["english"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )
    
    labels = tokenizer(
        example["hindi"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

dataset = dataset.map(preprocess, batched=True)
dataset.set_format(type="torch")


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [6]:
BATCH_SIZE = 4  # keep small for 4GB GPU

train_loader = DataLoader(
    dataset["train"],
    batch_size=BATCH_SIZE,
    shuffle=True
)


In [7]:
def add_noise(x, noise_level):
    noise = torch.randn_like(x)
    return x + noise_level * noise, noise
def get_noise_level(t, T):
    return t / T


In [9]:
class TinyDenoiser(nn.Module):
    def __init__(self, hidden_size=512):
        super().__init__()
        
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(
                d_model=hidden_size,
                nhead=8,
                dim_feedforward=1024,
                batch_first=True
            ),
            num_layers=2
        )
        
        self.output_layer = nn.Linear(hidden_size, hidden_size)

    def forward(self, x):
        x = self.transformer(x)
        return self.output_layer(x)


In [10]:
hidden_size = base_model.config.d_model  # 512 for mt5-small
denoiser = TinyDenoiser(hidden_size).to(device)


In [11]:
for param in base_model.parameters():
    param.requires_grad = False

embedding_layer = base_model.get_input_embeddings()


In [14]:
optimizer = torch.optim.Adam(denoiser.parameters(), lr=3e-4)
criterion = nn.MSELoss()

EPOCHS = 10
T = 10  # diffusion steps


In [15]:
for epoch in range(EPOCHS):
    total_loss = 0
    
    for batch in tqdm(train_loader):
        input_ids = batch["labels"].to(device)  # target Hindi
        
        with torch.no_grad():
            clean_embeddings = embedding_layer(input_ids)
        
        # sample random timestep
        t = torch.randint(1, T+1, (1,)).item()
        noise_level = get_noise_level(t, T)
        
        noisy_embeddings, noise = add_noise(clean_embeddings, noise_level)
        
        predicted = denoiser(noisy_embeddings)
        
        loss = criterion(predicted, clean_embeddings)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")


100%|██████████| 125/125 [00:01<00:00, 64.33it/s]


Epoch 1 | Loss: 72.4069


100%|██████████| 125/125 [00:01<00:00, 65.38it/s]


Epoch 2 | Loss: 64.7565


100%|██████████| 125/125 [00:01<00:00, 64.64it/s]


...

Epoch 8 | Loss: 38.2766


100%|██████████| 125/125 [00:01<00:00, 120.85it/s]


Epoch 9 | Loss: 35.6037


100%|██████████| 125/125 [00:01<00:00, 120.26it/s]

Epoch 10 | Loss: 33.3389


In [16]:
def test_translation(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    ).to(device)
    
    with torch.no_grad():
        embeddings = embedding_layer(inputs["input_ids"])
        noisy, _ = add_noise(embeddings, 0.5)
        denoised = denoiser(noisy)
        
        # project back to vocab space
        logits = torch.matmul(
            denoised,
            embedding_layer.weight.T
        )
        
        tokens = torch.argmax(logits, dim=-1)
        output = tokenizer.decode(tokens[0], skip_special_tokens=True)
    
    return output


In [18]:
print(test_translation("नमस्ते"))



नमस्ते
